# Instrumental magnitude analysis

The goal is to investigate random variations of instrumental magnitudes.
1. Is there measurement error for stars in the same frame totally random 
    or it's composed of common bias and random noise? 

First select a session and load photometry

In [ ]:
import vsopy.util
from vsopy.phot import BatchDataProvider

OBJ_NAME='SZ_Lyn'
SESSION_TAG='2025/20250126'
IMAGE_ROOT = '/srv/public/img'
WORK_ROOT = '/srv/public'

session = vsopy.util.Session(tag=SESSION_TAG, name=OBJ_NAME)
layout = vsopy.util.WorkLayout(WORK_ROOT)
session_layout = layout.get_session(session)
provider = BatchDataProvider(session_layout)


Remove target photometry (mean magnitude makes no sense for variale star).
Mow we have magnitudes $M_{i, s}$ for every star $s$ and image $i$  
and calculate average instrumental magnitude per star and band as well as stdev
for every star $s$ and band $b$

$$\mu_{s, b} \\ \sigma_{s, b}$$

In [ ]:
from astropy.table import join
import numpy as np

photometry = join(provider.images_['image_id', 'filter'], provider.measured_['image_id', 'auid', 'M'])['image_id', 'filter', 'auid', 'M']
photometry['M'] = photometry['M']['mag'].value
photometry = photometry[photometry['auid'] != provider.target_auid]
grouped = photometry['filter', 'auid', 'M'].group_by(['filter', 'auid'])
stats = join(grouped.groups.aggregate(np.mean), grouped.groups.aggregate(np.std), keys=['filter', 'auid'], table_names=['mean', 'std'])
stats


Deviation looks big.  We want to understand whether there is systemic bias
in each image.

Calculate magnitude residuals for each star and each image, taking imabre band
$b(i)$ into account, then normalize by standard deviation.:

$$ \rho_{i, s} = \frac{m_{i, s} - \mu_{s, b(i)}}{\sigma_{s, b(i)}} $$

 
Now we want to see whether residuals for stars in the same 
images are random or biased in the same direction.  
For this calculate average and  average norm of residuals for stars 
across image and divide first by second. Here $N(i)$ is the number of 
stars in the image.

$$ n_i = \frac{\sum\limits_{s} \rho_{i, s}}{ (\sqrt{\sum\limits_{s} r_{i, s}^2}) / N(i)} $$

 Call this normalised bias for the image.  The idea behind this value is 
 the following:  if residuals are random and uncorrelated, it should be 
 close to zero. If residuals are affected by common per-image offset, it should be
 close to $\pm1$.


In [ ]:

biases = join(photometry, stats, keys=['filter', 'auid'])
biases['res'] = (biases['M'] - biases['M_mean'])/biases['M_std']
biases = biases['image_id', 'auid', 'res']
biases_grouped = biases['image_id', 'res'].group_by('image_id')
norm_bias = join(biases_grouped.groups.aggregate(np.mean), biases_grouped.groups.aggregate(np.linalg.norm), keys='image_id', table_names=['mean', 'sq'])
norm_bias['size'] = np.diff(biases_grouped.groups.indices)
norm_bias['bias'] = norm_bias['res_mean'] / (norm_bias['res_sq'] / np.sqrt(norm_bias['size'])  )
norm_bias




 As we see, three are very few images with normalized bias below .7
 and in those cases normalized residual in average $(\sqrt{\sum\limits_{s} r_{i, s}^2}) / N(i)$
 is rather small, like .02 - .05 of standard deviation 

In [ ]:
norm_bias[np.abs(norm_bias['bias']) < .7]


Let's look into residuals for a single image

In [ ]:
rsd = join(photometry, stats, keys=['filter', 'auid'])
rsd['res'] = rsd['M'] - rsd['M_mean']
rsd['nres'] = rsd['res'] / rsd['M_std']

rsd[rsd['image_id'] == 42]

An important observation is that residual values in magnitudes looks close 
for al stars in the image.  It confirm the hypotesis that residuals are 
affected by common per-image factor (e.g transparency variation).  Note
that multiplicative changes in flux should result in additive changes in
magnitudes.

Having that said, let's separate offset and random part of the residual, so

$$ r_{i, s} = B_i + D_{i, s} $$

Given residuals for each star in the image

$$ r_{i, s} = m_{i, s} - \mu_{s, b(i)} $$

calculate bias value for the image as the average of residuals

$$ B_i = \frac{\sum\limits_{s} r_{i, s}}{N(i)} $$

then for each star we have noise component in magnitude as

$$ D_{i, s} = r_{i, s} - B_i $$

In [ ]:
# rsd = join(photometry, stats, keys=['filter', 'auid'])
# rsd['res'] = rsd['M'] - rsd['M_mean']
offset = rsd['image_id', 'res'].group_by('image_id').groups.aggregate(np.mean)
offset.rename_column('res', 'offset')
offset[30:50]


In [ ]:
noise = join(rsd, offset, keys='image_id')
noise['noise'] = noise['res'] - noise['offset']
noise[noise['image_id'] == 42]['image_id', 'auid', 'M', 'res',  'noise']

We assume that this noise component is the quality indicator beacuse it 
affects the quality of photometric transform (higher noise increases uncertainity
of transformed magnitudes).  Since it's noise, it should be aggregated as
L2 norm (square root of sum of squares).  It can be used to estimate the 
quality of a paticular image, band, or star.

Let's try per star metric first. Cleanly the star 000-BBP-105 should not be used.

In [ ]:
noise_grouped_by_star = noise['auid', 'noise'].group_by('auid')
noise_agg_by_star = noise_grouped_by_star.groups.aggregate(np.linalg.norm)
noise_agg_by_star['noise'] = noise_agg_by_star['noise'] / np.sqrt(np.diff(noise_grouped_by_star.groups.indices))
noise_agg_by_star


In [ ]:
noise_grouped_by_band = noise['filter', 'noise'].group_by('filter')
noise_agg_by_band = noise_grouped_by_band.groups.aggregate(np.linalg.norm)
noise_agg_by_band['noise'] = noise_agg_by_band['noise'] / np.sqrt(np.diff(noise_grouped_by_band.groups.indices))
noise_agg_by_band

In [ ]:
noise_grouped_by_star_band = noise['auid', 'filter', 'noise'].group_by(['auid', 'filter'])
noise_agg_by_star_band = noise_grouped_by_star_band.groups.aggregate(np.linalg.norm)
noise_agg_by_star_band['noise'] = noise_agg_by_star_band['noise'] / np.sqrt(np.diff(noise_grouped_by_star_band.groups.indices))
noise_agg_by_star_band.to_pandas().pivot(index='auid', columns='filter', values='noise')


In [ ]:
noise_grouped_by_image = noise['image_id', 'noise'].group_by('image_id')
noise_agg_by_image = noise_grouped_by_image.groups.aggregate(np.linalg.norm)
noise_agg_by_image['noise'] = noise_agg_by_image['noise'] / np.sqrt(np.diff(noise_grouped_by_image.groups.indices))
data_noise = join(join(join(provider.images_['image_id', 'filter'],  noise_agg_by_image), provider.batch_images_), provider.batches_)['image_id', 'filter', 'time', 'airmass', 'noise']
data_noise

In [ ]:
from plotnine import ggplot, aes, geom_point, geom_line, facet_grid, labs, theme, scale_color_manual, scale_x_datetime

ggplot(data_noise, aes(x='time', y='noise', color='filter')) + \
    geom_point() + \
    facet_grid('filter ~ .') + \
    scale_color_manual(values=['blue', 'magenta', 'red', 'green']) + \
    scale_x_datetime(date_labels='%Y-%m-%d %H:%M') + \
    labs(title=f'Instrumental Magnitude Noise for {OBJ_NAME} Session {SESSION_TAG}', x='UTC', y='Noise') + \
    theme(figure_size=(10, 10))

In [ ]:
ggplot(data_noise, aes(x='airmass', y='noise', color='filter')) + \
    geom_point() + \
    facet_grid('filter ~ .') + \
    scale_color_manual(values=['blue', 'magenta', 'red', 'green']) + \
    labs(title=f'Instrumental Magnitude Noise for {OBJ_NAME} Session {SESSION_TAG}', x='Airmass', y='Noise') + \
    theme(figure_size=(10, 10))

For completness plot offset behavior during the session

In [ ]:
data_offset = join(join(join(provider.images_['image_id', 'filter'],  offset), provider.batch_images_), provider.batches_)['image_id', 'filter', 'time', 'airmass', 'offset']

ggplot(data_offset, aes(x='time', y='offset', color='filter')) + \
    geom_point() + \
    facet_grid('filter ~ .') + \
    scale_color_manual(values=['blue', 'magenta', 'red', 'green']) + \
    scale_x_datetime(date_labels="%H:%M") + \
    labs(title=f'Instrumental Magnitude Offset for {OBJ_NAME} Session {SESSION_TAG}', x='Time', y='Offset') + \
    theme(figure_size=(10, 10))

In [ ]:
ggplot(data_offset, aes(x='airmass', y='offset', color='filter')) + \
    geom_point() + \
    facet_grid('filter ~ .') + \
    scale_color_manual(values=['blue', 'magenta', 'red', 'green']) + \
    labs(title=f'Instrumental Magnitude Offset for {OBJ_NAME} Session {SESSION_TAG}', x='Airmass', y='Offset') + \
    theme(figure_size=(10, 10))